<a href="https://colab.research.google.com/github/Nahmadid/SpectralBias/blob/main/gif_spctral_bias_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# === 1. Target function with multiple frequencies ===
def target_function(t):
    return jnp.sin(2 * jnp.pi * 0.01 * t) + 0.5 * jnp.sin(2 * jnp.pi * 0.05 * t) + 0.2 * jnp.sin(2 * jnp.pi * 0.1 * t)

# === 2. Chebyshev recursive basis ===
def chebyshev_recursive(x, degree):
    T_n_minus_2 = x * 0 + 1
    T_n_minus_1 = x
    result = [T_n_minus_2, T_n_minus_1]
    for n in range(2, degree + 1):
        T_n = 2 * x * T_n_minus_1 - T_n_minus_2
        result.append(T_n)
        T_n_minus_2, T_n_minus_1 = T_n_minus_1, T_n
    return jnp.stack(result, axis=-1)

# === 3. Gated cPIKAN initializer ===
def init_params_kan2(layers, degree, key=jax.random.PRNGKey(123)):
    keys = jax.random.split(key, len(layers))
    params = []
    for i in range(len(layers) - 2):
        W = jax.random.normal(keys[i], shape=(layers[i], layers[i+1], degree + 1)) / jnp.sqrt(layers[i] * (degree + 1))
        g = jax.random.normal(keys[i], shape=(layers[i+1],))
        params.append({'W': W, 'g': g})
    W = jax.random.normal(keys[-1], shape=(layers[-2], layers[-1])) / jnp.sqrt(layers[-2])
    B = jax.random.normal(keys[-1], shape=(layers[-1],))
    params.append({'W': W, 'B': B})
    return params

# === 4. Forward pass ===
def fwd(params, t, activation=jax.nn.tanh):
    t = 0.01 * t
    X = t.reshape((-1, 1))
    *hidden, last = params
    for layer in hidden:
        W = layer['W']
        g = layer['g']
        degree = W.shape[-1] - 1
        X_stack = chebyshev_recursive(X, degree)
        X = jnp.einsum("bid,iod->bo", X_stack, W)
        X = g * X
        X = activation(X)
    return X @ last['W'] + last['B']

# === 5. Loss ===
def mse_loss(params, t, y_true):
    y_pred = fwd(params, t)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)

# === 6. FFT Spectrum ===
def compute_fourier_spectrum(signal, dt):
    n = len(signal)
    freqs = np.fft.fftfreq(n, d=dt)
    fft_vals = np.fft.fft(signal)
    magnitude = np.abs(fft_vals)
    return freqs[:n // 2], magnitude[:n // 2]

# === 7. Setup ===
layers = [1, 64, 64, 1]
degree = 5
lr = 1e-4
epochs = 40000
log_epochs = list(range(0, epochs + 1, 1000))

t = jnp.linspace(0, 300, 301)
y_true = target_function(t)

params = init_params_kan2(layers, degree)
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)

# === 8. Training step ===
@jax.jit
def train_step(params, opt_state, t, y_true):
    loss, grads = jax.value_and_grad(mse_loss)(params, t, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# === 9. Training loop ===
predictions = {}
loss_history = []

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state, t, y_true)
    loss_history.append(loss)
    if epoch in log_epochs:
        predictions[epoch] = np.array(fwd(params, t)[:, 0])

# === 10. Animation: Signal vs. Prediction + Spectrum ===
plt.switch_backend("Agg")
dt = float(t[1] - t[0])
log_epochs = sorted(predictions.keys())

freqs = np.fft.fftfreq(len(t), d=dt)
mask = freqs >= 0
freqs_pos = freqs[mask]
fft_true = np.fft.fft(np.array(y_true))
amp_true = np.abs(fft_true)[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = log_epochs[i]
    y_pred = predictions[epoch]
    amp_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(t, y_true, label='True', color='black')
    axs[0].plot(t, y_pred, '--', label=f'Pred', color='red')
    axs[0].set_title(f"Signal Prediction at Epoch {epoch}")
    axs[0].set_xlabel("Time")
    axs[0].set_ylabel("Signal")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, amp_true, label='True Spectrum', color='black')
    axs[1].plot(freqs_pos, amp_pred, '--', label='Predicted Spectrum', color='red')
    axs[1].set_title(f"Frequency Spectrum at Epoch {epoch}")
    axs[1].set_xlabel("Frequency (Hz)")
    axs[1].set_ylabel("Amplitude")
    axs[1].set_xlim(0, 0.2)
    axs[1].set_yscale("log")
    axs[1].grid(True, which="both", ls="--", alpha=0.5)
    axs[1].legend()

anim = FuncAnimation(fig, animate, frames=len(log_epochs), interval=500)

# Save animation
anim.save("training_evolution_gkan.gif", writer=PillowWriter(fps=2))
anim.save("training_evolution_gkan.mp4", fps=2, extra_args=['-vcodec', 'libx264'])

print("Saved animation: training_evolution.gif, training_evolution.mp4")


Saved animation: training_evolution.gif, training_evolution.mp4


In [ ]:

anim.save("gkan_disc.gif", writer=PillowWriter(fps=2))

In [ ]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# === 1. Target function with multiple frequencies ===
def target_function(t):
    return jnp.sin(2 * jnp.pi * 0.01 * t) + 0.5 * jnp.sin(2 * jnp.pi * 0.05 * t) + 0.2 * jnp.sin(2 * jnp.pi * 0.1 * t)

# === 2. Chebyshev recursive basis ===
def chebyshev_recursive(x, degree):
    T_n_minus_2 = x * 0 + 1
    T_n_minus_1 = x
    result = [T_n_minus_2, T_n_minus_1]
    for n in range(2, degree + 1):
        T_n = 2 * x * T_n_minus_1 - T_n_minus_2
        result.append(T_n)
        T_n_minus_2, T_n_minus_1 = T_n_minus_1, T_n
    return jnp.stack(result, axis=-1)

# === 3. Gated cPIKAN initializer ===
def init_params_kan2(layers, degree, key=jax.random.PRNGKey(123)):
    keys = jax.random.split(key, len(layers))
    params = []
    for i in range(len(layers) - 2):
        W = jax.random.normal(keys[i], shape=(layers[i], layers[i+1], degree + 1)) / jnp.sqrt(layers[i] * (degree + 1))
        g = jax.random.normal(keys[i], shape=(layers[i+1],))
        params.append({'W': W, 'g': g})
    W = jax.random.normal(keys[-1], shape=(layers[-2], layers[-1])) / jnp.sqrt(layers[-2])
    B = jax.random.normal(keys[-1], shape=(layers[-1],))
    params.append({'W': W, 'B': B})
    return params

# === 4. Forward pass ===
def fwd(params, t, activation=jax.nn.tanh):
    t = 0.01 * t
    X = t.reshape((-1, 1))
    *hidden, last = params
    for layer in hidden:
        W = layer['W']
        g = layer['g']
        degree = W.shape[-1] - 1
        X_stack = chebyshev_recursive(X, degree)
        X = activation(X)
        X = jnp.einsum("bid,iod->bo", X_stack, W)
        # X = g * X
        X = activation(X)
    return X @ last['W'] + last['B']

# === 5. Loss ===
def mse_loss(params, t, y_true):
    y_pred = fwd(params, t)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)

# === 6. FFT Spectrum ===
def compute_fourier_spectrum(signal, dt):
    n = len(signal)
    freqs = np.fft.fftfreq(n, d=dt)
    fft_vals = np.fft.fft(signal)
    magnitude = np.abs(fft_vals)
    return freqs[:n // 2], magnitude[:n // 2]

# === 7. Setup ===
layers = [1, 64, 64, 1]
degree = 5
lr = 1e-4
epochs = 40000
log_epochs = list(range(0, epochs + 1, 1000))

t = jnp.linspace(0, 300, 301)
y_true = target_function(t)

params = init_params_kan2(layers, degree)
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)

# === 8. Training step ===
@jax.jit
def train_step(params, opt_state, t, y_true):
    loss, grads = jax.value_and_grad(mse_loss)(params, t, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# === 9. Training loop ===
predictions = {}
loss_history = []

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state, t, y_true)
    loss_history.append(loss)
    if epoch in log_epochs:
        predictions[epoch] = np.array(fwd(params, t)[:, 0])

# === 10. Animation: Signal vs. Prediction + Spectrum ===
plt.switch_backend("Agg")
dt = float(t[1] - t[0])
log_epochs = sorted(predictions.keys())

freqs = np.fft.fftfreq(len(t), d=dt)
mask = freqs >= 0
freqs_pos = freqs[mask]
fft_true = np.fft.fft(np.array(y_true))
amp_true = np.abs(fft_true)[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = log_epochs[i]
    y_pred = predictions[epoch]
    amp_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(t, y_true, label='True', color='black')
    axs[0].plot(t, y_pred, '--', label=f'Pred', color='red')
    axs[0].set_title(f"Signal Prediction at Epoch {epoch}")
    axs[0].set_xlabel("Time")
    axs[0].set_ylabel("Signal")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, amp_true, label='True Spectrum', color='black')
    axs[1].plot(freqs_pos, amp_pred, '--', label='Predicted Spectrum', color='red')
    axs[1].set_title(f"Frequency Spectrum at Epoch {epoch}")
    axs[1].set_xlabel("Frequency (Hz)")
    axs[1].set_ylabel("Amplitude")
    axs[1].set_xlim(0, 0.2)
    axs[1].set_yscale("log")
    axs[1].grid(True, which="both", ls="--", alpha=0.5)
    axs[1].legend()

anim = FuncAnimation(fig, animate, frames=len(log_epochs), interval=500)

# Save animation
anim.save("training_evolution_tanhcPIKAN.gif", writer=PillowWriter(fps=2))
anim.save("training_evolution_tanhcPIKAN.mp4", fps=2, extra_args=['-vcodec', 'libx264'])

print("Saved animation: training_evolution.gif, training_evolution.mp4")


Saved animation: training_evolution.gif, training_evolution.mp4


In [ ]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# === 1. Target function with multiple frequencies ===
def target_function(t):
    return jnp.sin(2 * jnp.pi * 0.01 * t) + 0.5 * jnp.sin(2 * jnp.pi * 0.05 * t) + 0.2 * jnp.sin(2 * jnp.pi * 0.1 * t)

# === 2. Chebyshev recursive basis ===
def chebyshev_recursive(x, degree):
    T_n_minus_2 = x * 0 + 1
    T_n_minus_1 = x
    result = [T_n_minus_2, T_n_minus_1]
    for n in range(2, degree + 1):
        T_n = 2 * x * T_n_minus_1 - T_n_minus_2
        result.append(T_n)
        T_n_minus_2, T_n_minus_1 = T_n_minus_1, T_n
    return jnp.stack(result, axis=-1)

# === 3. Gated cPIKAN initializer ===
def init_params_kan2(layers, degree, key=jax.random.PRNGKey(123)):
    keys = jax.random.split(key, len(layers))
    params = []
    for i in range(len(layers) - 2):
        W = jax.random.normal(keys[i], shape=(layers[i], layers[i+1], degree + 1)) / jnp.sqrt(layers[i] * (degree + 1))
        g = jax.random.normal(keys[i], shape=(layers[i+1],))
        params.append({'W': W, 'g': g})
    W = jax.random.normal(keys[-1], shape=(layers[-2], layers[-1])) / jnp.sqrt(layers[-2])
    B = jax.random.normal(keys[-1], shape=(layers[-1],))
    params.append({'W': W, 'B': B})
    return params

# === 4. Forward pass ===
def fwd(params, t, activation=jax.nn.tanh):
    t = 0.01 * t
    X = t.reshape((-1, 1))
    *hidden, last = params
    for layer in hidden:
        W = layer['W']
        g = layer['g']
        degree = W.shape[-1] - 1
        X = activation(X)
        X_stack = chebyshev_recursive(X, degree)
        X = jnp.einsum("bid,iod->bo", X_stack, W)
    if X.shape[1] > 1:
        X = X[:, 0:1]
        # X = g * X
    return X# @ last['W'] #+ last['B']

# === 5. Loss ===
def mse_loss(params, t, y_true):
    y_pred = fwd(params, t)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)

# === 6. FFT Spectrum ===
def compute_fourier_spectrum(signal, dt):
    n = len(signal)
    freqs = np.fft.fftfreq(n, d=dt)
    fft_vals = np.fft.fft(signal)
    magnitude = np.abs(fft_vals)
    return freqs[:n // 2], magnitude[:n // 2]

# === 7. Setup ===
layers = [1, 64, 64, 1]
degree = 5
lr = 1e-4
epochs = 40000
log_epochs = list(range(0, epochs + 1, 1000))

t = jnp.linspace(0, 300, 301)
y_true = target_function(t)

params = init_params_kan2(layers, degree)
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)

# === 8. Training step ===
@jax.jit
def train_step(params, opt_state, t, y_true):
    loss, grads = jax.value_and_grad(mse_loss)(params, t, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# === 9. Training loop ===
predictions = {}
loss_history = []

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state, t, y_true)
    loss_history.append(loss)
    if epoch in log_epochs:
        predictions[epoch] = np.array(fwd(params, t)[:, 0])

# === 10. Animation: Signal vs. Prediction + Spectrum ===
plt.switch_backend("Agg")
dt = float(t[1] - t[0])
log_epochs = sorted(predictions.keys())

freqs = np.fft.fftfreq(len(t), d=dt)
mask = freqs >= 0
freqs_pos = freqs[mask]
fft_true = np.fft.fft(np.array(y_true))
amp_true = np.abs(fft_true)[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = log_epochs[i]
    y_pred = predictions[epoch]
    amp_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(t, y_true, label='True', color='black')
    axs[0].plot(t, y_pred, '--', label=f'Pred', color='red')
    axs[0].set_title(f"Signal Prediction at Epoch {epoch}")
    axs[0].set_xlabel("Time")
    axs[0].set_ylabel("Signal")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, amp_true, label='True Spectrum', color='black')
    axs[1].plot(freqs_pos, amp_pred, '--', label='Predicted Spectrum', color='red')
    axs[1].set_title(f"Frequency Spectrum at Epoch {epoch}")
    axs[1].set_xlabel("Frequency (Hz)")
    axs[1].set_ylabel("Amplitude")
    axs[1].set_xlim(0, 0.2)
    axs[1].set_yscale("log")
    axs[1].grid(True, which="both", ls="--", alpha=0.5)
    axs[1].legend()

anim = FuncAnimation(fig, animate, frames=len(log_epochs), interval=500)

# Save animation
anim.save("training_evolution_cKAN.gif", writer=PillowWriter(fps=2))
anim.save("training_evolution_cKAN.mp4", fps=2, extra_args=['-vcodec', 'libx264'])

print("Saved animation: training_evolution.gif, training_evolution.mp4")


Saved animation: training_evolution.gif, training_evolution.mp4


In [ ]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# === 1. Piecewise target function ===
def target_function(x):
    left = 5.0 + jnp.sum(jnp.stack([jnp.sin(k * x) for k in range(1, 5)]), axis=0)
    right = jnp.cos(10 * x)
    return jnp.where(x < 0, left, right)

# === 2. Chebyshev recursive basis ===
def chebyshev_recursive(x, degree):
    T_n_minus_2 = x * 0 + 1
    T_n_minus_1 = x
    result = [T_n_minus_2, T_n_minus_1]
    for n in range(2, degree + 1):
        T_n = 2 * x * T_n_minus_1 - T_n_minus_2
        result.append(T_n)
        T_n_minus_2, T_n_minus_1 = T_n_minus_1, T_n
    return jnp.stack(result, axis=-1)

# === 3. KAN parameter initialization ===
def init_params_kan2(layers, degree, key=jax.random.PRNGKey(123)):
    keys = jax.random.split(key, len(layers))
    params = []
    for i in range(len(layers) - 2):
        W = jax.random.normal(keys[i], shape=(layers[i], layers[i+1], degree + 1)) / jnp.sqrt(layers[i] * (degree + 1))
        g = jax.random.normal(keys[i], shape=(layers[i+1],))
        params.append({'W': W, 'g': g})
    W = jax.random.normal(keys[-1], shape=(layers[-2], layers[-1])) / jnp.sqrt(layers[-2])
    B = jax.random.normal(keys[-1], shape=(layers[-1],))
    params.append({'W': W, 'B': B})
    return params

# === 4. Forward pass ===
def fwd(params, x, activation=jax.nn.tanh):
    X = x.reshape((-1, 1))
    *hidden, last = params
    for layer in hidden:
        W = layer['W']
        g = layer['g']
        degree = W.shape[-1] - 1
        X_stack = chebyshev_recursive(X, degree)
        X = jnp.einsum("bid,iod->bo", X_stack, W)
        X = g * X
        X = activation(X)
    return X @ last['W'] + last['B']

# === 5. MSE loss ===
def mse_loss(params, x, y_true):
    y_pred = fwd(params, x)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)

# === 6. FFT Spectrum ===
def compute_fourier_spectrum(signal, dx):
    n = len(signal)
    freqs = np.fft.fftfreq(n, d=dx)
    fft_vals = np.fft.fft(signal)
    magnitude = np.abs(fft_vals)
    return freqs[:n // 2], magnitude[:n // 2]

# === 7. Setup ===
layers = [1, 64, 64, 1]
degree = 5
lr = 1e-4
epochs = 40000
log_epochs = list(range(0, epochs + 1, 1000))

x = jnp.linspace(-jnp.pi, jnp.pi, 80)  # smaller dataset, centered domain
y_true = target_function(x)

params = init_params_kan2(layers, degree)
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)

# === 8. Training step ===
@jax.jit
def train_step(params, opt_state, x, y_true):
    loss, grads = jax.value_and_grad(mse_loss)(params, x, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# === 9. Training loop ===
predictions = {}
loss_history = []

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state, x, y_true)
    loss_history.append(loss)
    if epoch in log_epochs:
        predictions[epoch] = np.array(fwd(params, x)[:, 0])

# === 10. Animation: Signal vs. Prediction + Spectrum ===
plt.switch_backend("Agg")
dx = float(x[1] - x[0])
log_epochs = sorted(predictions.keys())

freqs = np.fft.fftfreq(len(x), d=dx)
mask = freqs >= 0
freqs_pos = freqs[mask]
fft_true = np.fft.fft(np.array(y_true))
amp_true = np.abs(fft_true)[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = log_epochs[i]
    y_pred = predictions[epoch]
    amp_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(x, y_true, label='True', color='black')
    axs[0].plot(x, y_pred, '--', label=f'Pred', color='red')
    axs[0].set_title(f"Signal Prediction at Epoch {epoch}")
    axs[0].set_xlabel("x")
    axs[0].set_ylabel("y")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, amp_true, label='True Spectrum', color='black')
    axs[1].plot(freqs_pos, amp_pred, '--', label='Predicted Spectrum', color='red')
    axs[1].set_title(f"Frequency Spectrum at Epoch {epoch}")
    axs[1].set_xlabel("f")
    axs[1].set_ylabel("Amplitude")
    axs[1].set_xlim(0, 6)
    axs[1].set_yscale("log")
    axs[1].grid(True, which="both", ls="--", alpha=0.5)
    axs[1].legend()

anim = FuncAnimation(fig, animate, frames=len(log_epochs), interval=500)

anim.save("gkan_disc.gif", writer=PillowWriter(fps=2))
print("Saved animation: gkan_disc.gif")


Saved animation: training_evolution_piecewise.gif


In [ ]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# === 1. Piecewise target function ===
def target_function(x):
    left = 5.0 + jnp.sum(jnp.stack([jnp.sin(k * x) for k in range(1, 5)]), axis=0)
    right = jnp.cos(10 * x)
    return jnp.where(x < 0, left, right)

# === 2. Chebyshev recursive basis ===
def chebyshev_recursive(x, degree):
    T_n_minus_2 = x * 0 + 1
    T_n_minus_1 = x
    result = [T_n_minus_2, T_n_minus_1]
    for n in range(2, degree + 1):
        T_n = 2 * x * T_n_minus_1 - T_n_minus_2
        result.append(T_n)
        T_n_minus_2, T_n_minus_1 = T_n_minus_1, T_n
    return jnp.stack(result, axis=-1)

# === 3. KAN parameter initialization ===
def init_params_kan2(layers, degree, key=jax.random.PRNGKey(123)):
    keys = jax.random.split(key, len(layers))
    params = []
    for i in range(len(layers) - 2):
        W = jax.random.normal(keys[i], shape=(layers[i], layers[i+1], degree + 1)) / jnp.sqrt(layers[i] * (degree + 1))
        g = jax.random.normal(keys[i], shape=(layers[i+1],))
        params.append({'W': W, 'g': g})
    W = jax.random.normal(keys[-1], shape=(layers[-2], layers[-1])) / jnp.sqrt(layers[-2])
    B = jax.random.normal(keys[-1], shape=(layers[-1],))
    params.append({'W': W, 'B': B})
    return params

# === 4. Forward pass ===
def fwd(params, x, activation=jax.nn.tanh):
    X = x.reshape((-1, 1))
    *hidden, last = params
    for layer in hidden:
        W = layer['W']
        g = layer['g']
        degree = W.shape[-1] - 1
        X = activation(X)
        X_stack = chebyshev_recursive(X, degree)
        X = jnp.einsum("bid,iod->bo", X_stack, W)
        # X = g * X
        X = activation(X)
    return X @ last['W'] + last['B']

# === 5. MSE loss ===
def mse_loss(params, x, y_true):
    y_pred = fwd(params, x)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)

# === 6. FFT Spectrum ===
def compute_fourier_spectrum(signal, dx):
    n = len(signal)
    freqs = np.fft.fftfreq(n, d=dx)
    fft_vals = np.fft.fft(signal)
    magnitude = np.abs(fft_vals)
    return freqs[:n // 2], magnitude[:n // 2]

# === 7. Setup ===
layers = [1, 64, 64, 1]
degree = 5
lr = 1e-4
epochs = 40000
log_epochs = list(range(0, epochs + 1, 1000))

x = jnp.linspace(-jnp.pi, jnp.pi, 80)  # smaller dataset, centered domain
y_true = target_function(x)

params = init_params_kan2(layers, degree)
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)

# === 8. Training step ===
@jax.jit
def train_step(params, opt_state, x, y_true):
    loss, grads = jax.value_and_grad(mse_loss)(params, x, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# === 9. Training loop ===
predictions = {}
loss_history = []

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state, x, y_true)
    loss_history.append(loss)
    if epoch in log_epochs:
        predictions[epoch] = np.array(fwd(params, x)[:, 0])

# === 10. Animation: Signal vs. Prediction + Spectrum ===
plt.switch_backend("Agg")
dx = float(x[1] - x[0])
log_epochs = sorted(predictions.keys())

freqs = np.fft.fftfreq(len(x), d=dx)
mask = freqs >= 0
freqs_pos = freqs[mask]
fft_true = np.fft.fft(np.array(y_true))
amp_true = np.abs(fft_true)[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = log_epochs[i]
    y_pred = predictions[epoch]
    amp_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(x, y_true, label='True', color='black')
    axs[0].plot(x, y_pred, '--', label=f'Pred', color='red')
    axs[0].set_title(f"Signal Prediction at Epoch {epoch}")
    axs[0].set_xlabel("x")
    axs[0].set_ylabel("y")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, amp_true, label='True Spectrum', color='black')
    axs[1].plot(freqs_pos, amp_pred, '--', label='Predicted Spectrum', color='red')
    axs[1].set_title(f"Frequency Spectrum at Epoch {epoch}")
    axs[1].set_xlabel("f")
    axs[1].set_ylabel("Amplitude")
    axs[1].set_xlim(0, 6)
    axs[1].set_yscale("log")
    axs[1].grid(True, which="both", ls="--", alpha=0.5)
    axs[1].legend()

anim = FuncAnimation(fig, animate, frames=len(log_epochs), interval=500)

anim.save("tanh-ckan_disc.gif", writer=PillowWriter(fps=2))
print("Saved animation: tanh-kan_disc.gif")


Saved animation: tanh-kan_disc.gif


In [ ]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# === 1. Piecewise target function ===
def target_function(x):
    left = 5.0 + jnp.sum(jnp.stack([jnp.sin(k * x) for k in range(1, 5)]), axis=0)
    right = jnp.cos(10 * x)
    return jnp.where(x < 0, left, right)

# === 2. Chebyshev recursive basis ===
def chebyshev_recursive(x, degree):
    T_n_minus_2 = x * 0 + 1
    T_n_minus_1 = x
    result = [T_n_minus_2, T_n_minus_1]
    for n in range(2, degree + 1):
        T_n = 2 * x * T_n_minus_1 - T_n_minus_2
        result.append(T_n)
        T_n_minus_2, T_n_minus_1 = T_n_minus_1, T_n
    return jnp.stack(result, axis=-1)

# === 3. KAN parameter initialization ===
def init_params_kan2(layers, degree, key=jax.random.PRNGKey(123)):
    keys = jax.random.split(key, len(layers))
    params = []
    for i in range(len(layers) - 2):
        W = jax.random.normal(keys[i], shape=(layers[i], layers[i+1], degree + 1)) / jnp.sqrt(layers[i] * (degree + 1))
        g = jax.random.normal(keys[i], shape=(layers[i+1],))
        params.append({'W': W, 'g': g})
    W = jax.random.normal(keys[-1], shape=(layers[-2], layers[-1])) / jnp.sqrt(layers[-2])
    B = jax.random.normal(keys[-1], shape=(layers[-1],))
    params.append({'W': W, 'B': B})
    return params

# === 4. Forward pass ===
def fwd(params, t, activation=jax.nn.tanh):
    # t = 0.01 * t
    X = t.reshape((-1, 1))
    *hidden, last = params
    for layer in hidden:
        W = layer['W']
        g = layer['g']
        degree = W.shape[-1] - 1
        X = activation(X)
        X_stack = chebyshev_recursive(X, degree)
        X = jnp.einsum("bid,iod->bo", X_stack, W)
    if X.shape[1] > 1:
        X = X[:, 0:1]
        # X = g * X
    return X# @ last['W'] #+ last['B']

# === 5. MSE loss ===
def mse_loss(params, x, y_true):
    y_pred = fwd(params, x)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)

# === 6. FFT Spectrum ===
def compute_fourier_spectrum(signal, dx):
    n = len(signal)
    freqs = np.fft.fftfreq(n, d=dx)
    fft_vals = np.fft.fft(signal)
    magnitude = np.abs(fft_vals)
    return freqs[:n // 2], magnitude[:n // 2]

# === 7. Setup ===
layers = [1, 64, 64, 1]
degree = 5
lr = 1e-4
epochs = 40000
log_epochs = list(range(0, epochs + 1, 1000))

x = jnp.linspace(-jnp.pi, jnp.pi, 80)  # smaller dataset, centered domain
y_true = target_function(x)

params = init_params_kan2(layers, degree)
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)

# === 8. Training step ===
@jax.jit
def train_step(params, opt_state, x, y_true):
    loss, grads = jax.value_and_grad(mse_loss)(params, x, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# === 9. Training loop ===
predictions = {}
loss_history = []

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state, x, y_true)
    loss_history.append(loss)
    if epoch in log_epochs:
        predictions[epoch] = np.array(fwd(params, x)[:, 0])

# === 10. Animation: Signal vs. Prediction + Spectrum ===
plt.switch_backend("Agg")
dx = float(x[1] - x[0])
log_epochs = sorted(predictions.keys())

freqs = np.fft.fftfreq(len(x), d=dx)
mask = freqs >= 0
freqs_pos = freqs[mask]
fft_true = np.fft.fft(np.array(y_true))
amp_true = np.abs(fft_true)[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = log_epochs[i]
    y_pred = predictions[epoch]
    amp_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(x, y_true, label='True', color='black')
    axs[0].plot(x, y_pred, '--', label=f'Pred', color='red')
    axs[0].set_title(f"Signal Prediction at Epoch {epoch}")
    axs[0].set_xlabel("x")
    axs[0].set_ylabel("y")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, amp_true, label='True Spectrum', color='black')
    axs[1].plot(freqs_pos, amp_pred, '--', label='Predicted Spectrum', color='red')
    axs[1].set_title(f"Frequency Spectrum at Epoch {epoch}")
    axs[1].set_xlabel("f")
    axs[1].set_ylabel("Amplitude")
    axs[1].set_xlim(0, 6)
    axs[1].set_yscale("log")
    axs[1].grid(True, which="both", ls="--", alpha=0.5)
    axs[1].legend()

anim = FuncAnimation(fig, animate, frames=len(log_epochs), interval=500)

anim.save("ckan_disc.gif", writer=PillowWriter(fps=2))
print("Saved animation: ckan_disc.gif")


Saved animation: ckan_disc.gif


In [1]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# === 1. Target function with multiple frequencies ===
def target_function(t):
    return (
        jnp.sin(2 * jnp.pi * 0.01 * t)
        + 0.5 * jnp.sin(2 * jnp.pi * 0.05 * t)
        + 0.2 * jnp.sin(2 * jnp.pi * 0.1 * t)
    )

# ==========================================================
#                 SIMPLE MLP (REPLACES KAN)
# ==========================================================
def init_params_mlp(layers, key=jax.random.PRNGKey(0)):
    params = []
    keys = jax.random.split(key, len(layers))

    for i in range(len(layers) - 1):
        W = jax.random.normal(keys[i], (layers[i], layers[i+1])) / jnp.sqrt(layers[i])
        B = jnp.zeros((layers[i+1],))
        params.append({"W": W, "B": B})

    return params


def fwd_mlp(params, t, activation=jax.nn.tanh):
    x = 0.01 * t.reshape((-1, 1))

    for layer in params[:-1]:
        x = activation(x @ layer["W"] + layer["B"])

    last = params[-1]
    out = x @ last["W"] + last["B"]  # (N,1)
    return out


# === 3. Loss ===
def mse_loss(params, t, y_true):
    y_pred = fwd_mlp(params, t)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)


# === 4. FFT Spectrum ===
def compute_fourier_spectrum(signal, dt):
    n = len(signal)
    freqs = np.fft.fftfreq(n, d=dt)
    fft_vals = np.fft.fft(signal)
    magnitude = np.abs(fft_vals)
    return freqs[: n // 2], magnitude[: n // 2]


# === 5. Setup ===
layers = [1, 64, 64, 1]   # Standard MLP architecture
lr = 1e-4
epochs = 40000
log_epochs = list(range(0, epochs + 1, 1000))

t = jnp.linspace(0, 300, 301)
y_true = target_function(t)

params = init_params_mlp(layers)
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)


# === 6. Training step ===
@jax.jit
def train_step(params, opt_state, t, y_true):
    loss, grads = jax.value_and_grad(mse_loss)(params, t, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss


# === 7. Training loop ===
predictions = {}
loss_history = []

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state, t, y_true)
    loss_history.append(loss)
    if epoch in log_epochs:
        predictions[epoch] = np.array(fwd_mlp(params, t)[:, 0])


# === 8. Animation ===
plt.switch_backend("Agg")
dt = float(t[1] - t[0])
log_epochs = sorted(predictions.keys())

freqs = np.fft.fftfreq(len(t), d=dt)
mask = freqs >= 0
freqs_pos = freqs[mask]
amp_true = np.abs(np.fft.fft(np.array(y_true)))[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = log_epochs[i]
    y_pred = predictions[epoch]
    amp_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(t, y_true, label='True', color='black')
    axs[0].plot(t, y_pred, '--', label='Pred', color='red')
    axs[0].set_title(f"Signal Prediction at Epoch {epoch}")
    axs[0].set_xlabel("Time")
    axs[0].set_ylabel("Signal")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, amp_true, label='True Spectrum', color='black')
    axs[1].plot(freqs_pos, amp_pred, '--', label='Pred Spectrum', color='red')
    axs[1].set_title(f"Frequency Spectrum at Epoch {epoch}")
    axs[1].set_xlabel("Frequency")
    axs[1].set_ylabel("Amplitude (log)")
    axs[1].set_xlim(0, 0.2)
    axs[1].set_yscale("log")
    axs[1].grid(True, which="both", ls="--", alpha=0.5)
    axs[1].legend()

anim = FuncAnimation(fig, animate, frames=len(log_epochs), interval=500)

anim.save("training_evolution_mlp.gif", writer=PillowWriter(fps=2))
anim.save("training_evolution_mlp.mp4", fps=2, extra_args=['-vcodec', 'libx264'])

print("Saved animation: training_evolution_mlp.gif, training_evolution_mlp.mp4")


Saved animation: training_evolution_mlp.gif, training_evolution_mlp.mp4


In [3]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# === 1. Piecewise target function ===
def target_function(x):
    left = 5.0 + jnp.sum(jnp.stack([jnp.sin(k * x) for k in range(1, 5)]), axis=0)
    right = jnp.cos(10 * x)
    return jnp.where(x < 0, left, right)


# ==========================================================
#               SIMPLE MLP (REPLACES KAN)
# ==========================================================
def init_params_mlp(layers, key=jax.random.PRNGKey(0)):
    params = []
    keys = jax.random.split(key, len(layers))

    for i in range(len(layers) - 1):
        W = jax.random.normal(keys[i], (layers[i], layers[i+1])) / jnp.sqrt(layers[i])
        B = jnp.zeros((layers[i+1],))
        params.append({"W": W, "B": B})

    return params


def fwd(params, x, activation=jax.nn.tanh):
    X = x.reshape((-1, 1))

    for layer in params[:-1]:
        X = activation(X @ layer["W"] + layer["B"])

    last = params[-1]
    X = X @ last["W"] + last["B"]
    return X


# === 5. MSE loss ===
def mse_loss(params, x, y_true):
    y_pred = fwd(params, x)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)


# === 6. FFT Spectrum ===
def compute_fourier_spectrum(signal, dx):
    n = len(signal)
    freqs = np.fft.fftfreq(n, d=dx)
    fft_vals = np.fft.fft(signal)
    magnitude = np.abs(fft_vals)
    return freqs[:n // 2], magnitude[:n // 2]


# === 7. Setup ===
layers = [1, 32, 32, 1]
lr = 1e-4
epochs = 40000
log_epochs = list(range(0, epochs + 1, 1000))

x = jnp.linspace(-jnp.pi, jnp.pi, 80)
y_true = target_function(x)

params = init_params_mlp(layers)
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)


# === 8. Training step ===
@jax.jit
def train_step(params, opt_state, x, y_true):
    loss, grads = jax.value_and_grad(mse_loss)(params, x, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss


# === 9. Training loop ===
predictions = {}
loss_history = []

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state, x, y_true)
    loss_history.append(loss)
    if epoch in log_epochs:
        predictions[epoch] = np.array(fwd(params, x)[:, 0])


# === 10. Animation: Signal vs. Prediction + Spectrum ===
plt.switch_backend("Agg")
dx = float(x[1] - x[0])
log_epochs = sorted(predictions.keys())

freqs = np.fft.fftfreq(len(x), d=dx)
mask = freqs >= 0
freqs_pos = freqs[mask]

fft_true = np.fft.fft(np.array(y_true))
amp_true = np.abs(fft_true)[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = log_epochs[i]
    y_pred = predictions[epoch]
    amp_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(x, y_true, label='True', color='black')
    axs[0].plot(x, y_pred, '--', label=f'Pred', color='red')
    axs[0].set_title(f"Signal Prediction at Epoch {epoch}")
    axs[0].set_xlabel("x")
    axs[0].set_ylabel("y")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, amp_true, label='True Spectrum', color='black')
    axs[1].plot(freqs_pos, amp_pred, '--', label='Predicted Spectrum', color='red')
    axs[1].set_title(f"Frequency Spectrum at Epoch {epoch}")
    axs[1].set_xlabel("f")
    axs[1].set_ylabel("Amplitude")
    axs[1].set_xlim(0, 6)
    axs[1].set_yscale("log")
    axs[1].grid(True, which="both", ls="--", alpha=0.5)
    axs[1].legend()

anim = FuncAnimation(fig, animate, frames=len(log_epochs), interval=500)

anim.save("mlp_piecewise.gif", writer=PillowWriter(fps=2))
print("Saved animation: mlp_piecewise.gif")


Saved animation: mlp_piecewise.gif
